# External validation against reported ideology

This notebook asks how well DYNAMAP coordinates predict reported left–right identification. It fits four random-forest specifications on the **same participants and the same ten folds** in each country: coordinates, demographics, both together, and the first principal component of the coordinates.

**Inputs:** supplied participant coordinates in `outputs/models/`, original metadata in `data/raw/`, age decisions in `data/processed/`, and participant comparison counts. Notebooks 01–03 regenerate those inputs. You can run this notebook in a fresh kernel without running those notebooks first because their outputs are included.

**Outputs:** participant samples, out-of-fold predictions, fold metrics, ROC data, feature importances, and activity-group summaries in `outputs/validation/`. Notebooks 15 and 16 use these files for figures and tables.

Run all cells to refit the external classifiers. This fits 120 forests of 2,000 trees, so allow time for the analysis to finish. The random-forest, cross-validation and activity-bin settings are visible below. Changing them regenerates the validation outputs, but does not retrain DYNAMAP.

Labels are used only as external outcomes. Chilean positions 0–4 are left and 6–10 are right. French and Brazilian positions 1–2 are left and 4–5 are right. Central and nonresponse categories are omitted from this binary comparison.

People with unknown ages remain in the DYNAMAP maps. For this four-way comparison, however, all demographic predictors must be available: a known age of 19–120 in Chile or an age-category code of 2–7 in France and Brazil, plus the sex and region rules below. This restriction gives all four classifiers an identical evaluation sample.


In [1]:
from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import MinMaxScaler

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "data/raw").is_dir(), "Open from the project root or notebooks directory."

OUT = ROOT / "outputs/validation"
OUT.mkdir(parents=True, exist_ok=True)
COUNTRIES = ["Chile", "France", "Brazil"]
COORDINATES = {
    "Chile": "outputs/models/Chile_dim2_seed101/coordsU.csv",
    "France": "outputs/models/france/coordsU.csv",
    "Brazil": "outputs/models/brazil/coordsU.csv",
}
COUNTS = {
    "Chile": "data/processed/Chile_participant_support.csv",
    "France": "outputs/tables/france_participation_counts.csv",
    "Brazil": "outputs/tables/brazil_participation_counts.csv",
}
RF_PARAMETERS = {"n_estimators": 2000, "criterion": "entropy", "random_state": 42, "n_jobs": 2}
CV_PARAMETERS = {"n_splits": 10, "shuffle": True, "random_state": 42}
ACTIVITY_GROUPS = 10
DEMOGRAPHICS = {
    "Chile": ["age", "sex", "region"],
    "France": ["age", "sex"],
    "Brazil": ["age", "sex", "region"],
}

age_decisions = pd.read_csv(
    ROOT / "data/processed/participant_age_decisions.csv", dtype={"uuid": str}
)
assert not age_decisions.duplicated(["country", "uuid"]).any()
assert age_decisions.include.isin([True, False]).all()

# Match the original demographic coding, including alternate region spellings.
CHILE_REGION = {
    0: 0,
    "Arica y Parinacota": 1,
    "Tarapacá": 2,
    "Antofagasta": 3,
    "Atacama": 4,
    "Coquimbo": 5,
    "Valparaíso": 6,
    "Metropolitana de Santiago": 7,
    "Santiago Metropolitan": 7,
    "O'Higgins": 8,
    "Maule": 9,
    "Ñuble": 10,
    "Biobío": 11,
    "Biobio": 11,
    "Araucanía": 12,
    "Los Ríos": 13,
    "Los Lagos": 14,
    "Aysén": 15,
    "Magallanes": 16,
    "Outside of Chile": 17,
    "Fuera de Chile": 17,
}

BRAZIL_REGION = {
    "PR": 0,
    "RS": 1,
    "MG": 2,
    "PE": 3,
    "GO": 4,
    "SC": 5,
    "AC": 6,
    "MS": 7,
    "PB": 8,
    "AL": 9,
    "BA": 10,
    "RJ": 11,
    "SP": 12,
    "DF": 13,
    "SE": 14,
    "ES": 15,
    "RN": 16,
    "RO": 17,
    "CE": 18,
    "AM": 19,
    "RR": 20,
    "MA": 21,
    "PA": 22,
    "PI": 23,
}


## Build one unambiguous record per participant

The metadata join can contain multiple survey rows for a participant. Exact duplicates are harmless and are removed. If rows disagree on a predictor or the binary outcome, the entire ID is excluded rather than choosing one response. Missing or uncodable predictors also exclude the ID from this common validation sample. The exclusions and retained class counts are displayed below.

France uses `(z2, 1-z1)` to retain the original classifier column convention. This is a rigid rotation and translation: it changes neither Euclidean distances nor the information in the two coordinates. It uses no ideological labels and does not align independently fitted countries.


In [2]:
frames = {}
sample_rows = []
for country in COUNTRIES:
    destination = OUT / country
    destination.mkdir(parents=True, exist_ok=True)
    key = "uuid" if country == "Chile" else "user_id"
    metadata_path = ROOT / "data" / "raw" / f"{country.lower()}_metadata.csv"
    coordinate_path = ROOT / COORDINATES[country]
    metadata = pd.read_csv(metadata_path, dtype={key: str})
    coordinates = pd.read_csv(coordinate_path, dtype={key: str})
    assert coordinates[key].is_unique
    decisions = age_decisions[
        age_decisions.country.str.casefold() == country.casefold()
    ].set_index("uuid")
    assert set(coordinates[key]).issubset(decisions.index)
    assert decisions.loc[coordinates[key], "include"].all()
    # Each metadata row must match one coordinate row. Repeated survey rows are handled below.
    frame = pd.merge(metadata, coordinates, on=key, validate="many_to_one")
    left = [0, 1, 2, 3, 4] if country == "Chile" else [1, 2]
    right = [6, 7, 8, 9, 10] if country == "Chile" else [4, 5]
    frame = frame[frame.politica.isin(left + right)].copy()
    frame["target"] = frame.politica.isin(right).astype(int)
    demo = DEMOGRAPHICS[country]
    columns = demo + ["z1", "z2"]
    original_ids = frame[key].nunique()
    frame = frame[[key] + columns + ["target"]].drop_duplicates().copy()
    # Exclude ambiguous IDs before recoding to avoid selecting an arbitrary response.
    reasons = {}
    for identifier, group in frame.groupby(key, sort=False):
        differing = [
            col for col in columns + ["target"] if group[col].nunique(dropna=False) > 1
        ]
        missing = [col for col in columns + ["target"] if group[col].isna().any()]
        if differing or missing:
            reasons[identifier] = {"conflicting_fields": differing, "missing_fields": missing}
    clean = frame[~frame[key].isin(reasons)].copy()
    if country == "Chile":
        clean["sex"] = clean.sex.map({"Femenino": 0, "Masculino": 1, "Otro": 2})
        clean["region"] = clean.region.map(CHILE_REGION)
    elif country == "Brazil":
        clean["region"] = clean.region.map(BRAZIL_REGION)
    else:
        old_z1 = clean.z1.copy()
        clean["z1"] = clean.z2
        clean["z2"] = 1 - old_z1
    for col in columns:
        clean[col] = pd.to_numeric(clean[col], errors="coerce")
    # Coercion marks unknown region labels and nonnumeric predictors as missing.
    invalid = ~np.isfinite(clean[columns].to_numpy()).all(axis=1)
    for identifier in clean.loc[invalid, key]:
        reasons[identifier] = {
            "conflicting_fields": [],
            "missing_fields": ["uncodable_or_nonfinite_predictor"],
        }
    clean = clean.loc[~invalid].copy()
    # Unknown ages are valid for mapping, but cannot be demographic predictors here.
    eligibility_reasons = {}
    for _, row in clean.iterrows():
        fields = []
        if country == "Chile":
            if not 19 <= row.age <= 120:
                fields.append("known_age_required_19_120")
        else:
            if row.age not in range(2, 8):
                fields.append("known_age_required_codes_2_7")
            if row.sex not in [1, 2]:
                fields.append("sex_not_in_codes_1_2")
        if fields:
            eligibility_reasons[row[key]] = fields
            reasons[row[key]] = {
                "conflicting_fields": [],
                "missing_fields": [],
                "eligibility_fields": fields,
            }
    clean = clean[~clean[key].isin(eligibility_reasons)].copy()
    assert clean[key].is_unique
    clean = clean.rename(columns={key: "participant_id"}).reset_index(drop=True)
    assert clean.target.value_counts().min() >= CV_PARAMETERS["n_splits"]
    assert clean.participant_id.isin(coordinates[key]).all()
    display(clean)
    clean.to_csv(destination / "eligible_participants.csv", index=False)
    frames[country] = clean
    exclusion_table = pd.DataFrame(
        [
            {
                "participant_id": identifier,
                "conflicting_fields": ";".join(reason["conflicting_fields"]),
                "missing_fields": ";".join(reason["missing_fields"]),
                "eligibility_fields": ";".join(reason.get("eligibility_fields", [])),
            }
            for identifier, reason in reasons.items()
        ],
        columns=[
            "participant_id",
            "conflicting_fields",
            "missing_fields",
            "eligibility_fields",
        ],
    )
    display(exclusion_table)
    exclusion_table.to_csv(destination / "exclusions.csv", index=False)
    sample_rows.append(
        {
            "country": country,
            "map_participants": len(coordinates),
            "binary_label_ids_before_cleaning": int(original_ids),
            "excluded_from_common_sample": len(reasons),
            "common_sample": len(clean),
            "left": int(clean.target.eq(0).sum()),
            "right": int(clean.target.eq(1).sum()),
        }
    )

sample_summary = pd.DataFrame(sample_rows)
display(sample_summary)
sample_summary.to_csv(OUT / "validation_sample_audit.csv", index=False)
sample_summary.to_latex(OUT / "validation_sample_audit.tex", index=False)


,participant_id,age,sex,region,z1,z2,target
0,8bd42737-b140-4db8-af4c-59839922129a,27,1,11,0.277013,0.199440,0
1,7d2e034a-3270-4429-aeac-418c46bf0d7d,43,1,7,0.364785,0.264025,1
2,c1b97bb6-03d2-4027-8d54-0dd64a927a6c,39,0,7,0.490336,0.318748,0
3,883528e9-0599-4d30-9f4c-1daf9cf74082,34,1,7,0.429439,0.882237,1
4,7fd1210c-b1c0-4bdf-b190-7bfa10fe51d2,41,1,17,0.179798,0.743944,0
...,...,...,...,...,...,...,...
22019,759b16f2-f527-4f04-85ba-682fd1af8a89,41,1,9,0.177947,0.396998,0
22020,2e8f5550-d794-411c-b36e-d277e88f5d30,33,0,9,0.590559,0.113771,1
22021,3ddc9eae-ba7b-4dab-8e43-5c188daf9692,51,0,14,0.424433,0.334131,0
22022,b7132b2c-5ca4-44ac-b4e7-07e43d4b7547,33,1,7,0.473798,0.253386,1


,participant_id,conflicting_fields,missing_fields,eligibility_fields
0,d58cde44-5aa9-416f-9f1e-1f2841fac4b1,region,,
1,fc25f6e6-2351-499c-a3a4-3ab09694a869,target,,
2,1bdee467-b056-45dc-8c1d-6c47b591c89c,target,,
3,e012760e-08a4-47a6-bbf3-57a118c68925,target,,
4,87a5517a-f670-48d1-9850-bd7c4fde4025,sex,,
5,c5823f53-1b0c-4ad7-b804-50702fe37d6b,region,,
6,bcf58bd5-c7b3-4c86-87c2-bf36d3f5a29f,sex,,
7,d814cc9e-76e2-4ff1-86e9-0159601e6929,target,,
8,9ea04235-600d-4907-a6ba-b01e2ee1b501,target,,
9,84bb8768-cede-44cc-9b37-c59601ad430c,target,,


,participant_id,age,sex,z1,z2,target
0,e9001e65-7aa1-468d-a7c6-dca4ad023e59,2,1,6.933758e-02,0.374006,0
1,41e3bf3a-e792-4521-8d3a-8f974f329345,2,1,1.806362e-04,0.299299,0
2,a664ff56-e303-4ab5-8841-bc950222e75f,3,1,1.541898e-07,0.246321,0
3,7490cb32-18ba-4572-9310-c0c3801438d0,2,2,7.282050e-01,0.502114,1
4,fecf3440-84f3-4bcb-b2c4-f6c939d80ec7,2,1,1.193472e-04,0.327659,0
...,...,...,...,...,...,...
783,9026410e-9aa7-4bf8-aafa-8dbd26c94c63,2,2,7.172434e-01,0.456519,1
784,67e9a740-4137-4834-9331-a54929c5e6a9,5,1,1.640098e-01,0.244334,0
785,48243d8e-ab20-4379-b9ba-ba76c4cde2f3,3,1,2.499202e-01,0.386546,0
786,a7adb7e7-e526-49b9-93d5-563f75aa7714,2,2,3.109551e-03,0.205531,0


,participant_id,conflicting_fields,missing_fields,eligibility_fields
0,118fed35-ddfd-402d-888b-88bb3bffdbd0,sex,,
1,539b2141-4d8a-4ff0-890a-beac65df93b7,,,sex_not_in_codes_1_2
2,e5db20f0-0bc8-493e-85da-dfadf9e41421,,,sex_not_in_codes_1_2
3,a1653595-b08e-4ad0-927f-8c94f094019a,,,sex_not_in_codes_1_2
4,2c1c9180-6a55-47b9-b563-96ce2538bcbc,,,sex_not_in_codes_1_2
5,4ec52ca2-1fdb-48e9-8861-4f8306984c65,,,sex_not_in_codes_1_2
6,d16fbdf2-4618-406d-85f1-057acebce618,,,sex_not_in_codes_1_2
7,8a65e0a0-c707-4531-8db2-58f577d58685,,,sex_not_in_codes_1_2
8,7df72060-aa57-4c1d-bec7-8990861dea0f,,,sex_not_in_codes_1_2
9,9890b3cd-8e4a-4764-bcdb-271299fdc060,,,known_age_required_codes_2_7;sex_not_in_codes_1_2


,participant_id,age,sex,region,z1,z2,target
0,e3428c81-c1a3-4139-9deb-ab2b9aae481d,4,2,1,0.666261,0.503207,0
1,248860f8-a008-4207-84fd-a88cdc02e3f1,5,2,1,0.468704,0.514399,0
2,f3d2061a-9ffa-4fc2-bf46-f9990b6055ec,3,1,1,0.648949,0.380455,0
3,457cc0fc-957a-4984-a685-587b0686d18d,4,2,1,0.847903,0.614312,0
4,fc8481a9-7975-43f3-8a76-c054d51e73ea,3,1,1,0.335343,0.527585,1
...,...,...,...,...,...,...,...
490,33c490b5-f5e4-4db9-8d77-4f76fd0d363f,2,2,18,0.509419,0.426989,0
491,5d5f537f-a895-49ed-966d-a5b31539155d,5,1,12,0.277823,0.535905,1
492,035fef84-952f-40cf-b6bc-59419ebc4234,6,1,12,0.329012,0.429228,1
493,730666d4-bf22-4c25-9df0-cd447cd72100,6,2,4,0.360268,0.407338,1


,participant_id,conflicting_fields,missing_fields,eligibility_fields
0,adac3794-d0c1-4303-ab1b-aa79aec47a0b,,,sex_not_in_codes_1_2
1,0795ecd3-c623-464d-be92-f969664a6e11,,,sex_not_in_codes_1_2
2,3ba8a75e-1711-4c96-8c9b-7888e70fcdca,,,sex_not_in_codes_1_2
3,97752f8e-da1b-4963-b44f-b31c68635dda,,,sex_not_in_codes_1_2


,country,map_participants,binary_label_ids_before_cleaning,excluded_from_common_sample,common_sample,left,right
0,Chile,47574,22071,47,22024,12462,9562
1,France,1443,813,25,788,644,144
2,Brazil,765,499,4,495,373,122


Matplotlib is building the font cache; this may take a moment.


## Predict each participant with a forest trained on other participants

Each participant receives one out-of-fold prediction: the forest producing that prediction never sees their ideological label during training. The coordinates themselves were estimated from policy comparisons, including this participant's comparisons. This evaluates external ideological identification among people already in the map, not prediction for people with no observed policy choices.

Within each country, the fold assignment is computed once and reused by all four specifications. PCA and scaling are fitted on the training fold only. The classifier produces both a binary prediction and a probability of being on the right, which is used for the ROC curve and AUC.

The nested loops below run countries, predictor specifications, and folds in that order. Fold-level progress appears inline. A pooled out-of-fold accuracy weights participants equally, whereas mean fold accuracy weights the ten folds equally. Both are retained. Feature importances are the forest's impurity-based importances, not causal effects.


In [3]:
summary_rows = []
combined_predictions = {}
for country in COUNTRIES:
    frame = frames[country]
    destination = OUT / country
    target = frame.target.to_numpy()
    # Reuse this exact partition for every predictor specification.
    partitions = list(StratifiedKFold(**CV_PARAMETERS).split(frame, target))
    assignment = np.full(len(frame), -1)
    for fold, (_, test) in enumerate(partitions):
        assignment[test] = fold
    assert (assignment >= 0).all()
    assignments = pd.DataFrame(
        {"participant_id": frame.participant_id, "target": target, "fold": assignment}
    )
    display(assignments)
    assignments.to_csv(destination / "fold_assignments.csv", index=False)
    specifications = {
        "coordinates": ["z1", "z2"],
        "demographics": DEMOGRAPHICS[country],
        "combined": DEMOGRAPHICS[country] + ["z1", "z2"],
        "pc1": ["z1", "z2"],
    }
    summaries = {}
    for model_name, features in specifications.items():
        # Test indices write back into participant order, so each row is predicted once.
        probability = np.full(len(frame), np.nan)
        predicted = np.full(len(frame), -1)
        fold_metrics, importance = [], []
        input_matrix = frame[features].to_numpy(dtype=float)
        feature_names = ["PC1"] if model_name == "pc1" else features
        for fold, (train, test) in enumerate(partitions):
            assert not (
                set(frame.participant_id.iloc[train]) & set(frame.participant_id.iloc[test])
            )
            train_x, test_x = input_matrix[train], input_matrix[test]
            # Fit transformations on training participants, then apply to held-out rows.
            if model_name == "pc1":
                pca = PCA(n_components=1, svd_solver="full")
                train_x = pca.fit_transform(train_x)
                test_x = pca.transform(test_x)
            scaler = MinMaxScaler()
            train_x = scaler.fit_transform(train_x)
            test_x = scaler.transform(test_x)
            classifier = RandomForestClassifier(**RF_PARAMETERS)
            before = time.perf_counter()
            classifier.fit(train_x, target[train])
            probability[test] = classifier.predict_proba(test_x)[:, 1]
            predicted[test] = classifier.predict(test_x)
            metrics = {
                "fold": fold,
                "n_train": len(train),
                "n_test": len(test),
                "accuracy": float(accuracy_score(target[test], predicted[test])),
                "recall_left": float(recall_score(target[test], predicted[test], pos_label=0)),
                "recall_right": float(recall_score(target[test], predicted[test], pos_label=1)),
                "auc": float(roc_auc_score(target[test], probability[test])),
                "fit_seconds": time.perf_counter() - before,
            }
            fold_metrics.append(metrics)
            importance.append(dict(zip(feature_names, classifier.feature_importances_)))
            message = f"{country} {model_name}: fold {fold + 1}/{len(partitions)}, accuracy={metrics['accuracy']:.5f}, seconds={metrics['fit_seconds']:.1f}"
            print(message, flush=True)
        assert np.isfinite(probability).all() and (predicted >= 0).all()
        predictions = pd.DataFrame(
            {
                "participant_id": frame.participant_id,
                "target": target,
                "fold": assignment,
                "prediction": predicted,
                "probability_right": probability,
            }
        )
        pd.testing.assert_frame_equal(
            predictions[["participant_id", "target", "fold"]], assignments
        )
        if model_name == "combined":
            combined_predictions[country] = predictions.copy()
        display(predictions)
        predictions.to_csv(destination / f"{model_name}_oof_predictions.csv", index=False)
        fold_table = pd.DataFrame(fold_metrics)
        display(fold_table)
        fold_table.to_csv(destination / f"{model_name}_fold_metrics.csv", index=False)
        fold_table.to_latex(
            destination / f"{model_name}_fold_metrics.tex", index=False, float_format="%.6f"
        )
        importance_table = pd.DataFrame(importance).rename_axis("fold").reset_index()
        display(importance_table)
        importance_table.to_csv(
            destination / f"{model_name}_feature_importance_by_fold.csv", index=False
        )
        importance_table.to_latex(
            destination / f"{model_name}_feature_importance_by_fold.tex",
            index=False,
            float_format="%.6f",
        )
        fpr, tpr, thresholds = roc_curve(target, probability)
        curve = pd.DataFrame({"FPR": fpr, "TPR": tpr, "threshold": thresholds})
        display(curve)
        curve.to_csv(destination / f"{model_name}_roc.csv", index=False)
        scores = fold_table.accuracy.to_numpy()
        summary = {
            "features": feature_names,
            "fold_accuracy_min": float(scores.min()),
            "fold_accuracy_max": float(scores.max()),
            "fold_accuracy_mean": float(scores.mean()),
            "oof_accuracy": float(accuracy_score(target, predicted)),
            "oof_recall_left": float(recall_score(target, predicted, pos_label=0)),
            "oof_recall_right": float(recall_score(target, predicted, pos_label=1)),
            "oof_auc": float(roc_auc_score(target, probability)),
            "confusion_counts_left_right": confusion_matrix(
                target, predicted, labels=[0, 1]
            ).tolist(),
            "confusion_row_normalized_left_right": confusion_matrix(
                target, predicted, labels=[0, 1], normalize="true"
            ).tolist(),
            "mean_feature_importance": pd.DataFrame(importance).mean().to_dict(),
        }
        summaries[model_name] = summary
        summary_rows.append(
            {
                "country": country,
                "model": model_name,
                "n": len(frame),
                **{
                    key: summary[key]
                    for key in [
                        "fold_accuracy_min",
                        "fold_accuracy_max",
                        "fold_accuracy_mean",
                        "oof_accuracy",
                        "oof_recall_left",
                        "oof_recall_right",
                        "oof_auc",
                    ]
                },
            }
        )
    display(summaries)
    (destination / "metrics_summary.json").write_text(json.dumps(summaries, indent=2) + "\n")

summary_table = pd.DataFrame(summary_rows)
display(summary_table)
summary_table.to_csv(OUT / "validation_summary.csv", index=False)
summary_table.to_latex(OUT / "validation_summary.tex", index=False, float_format="%.6f")


,participant_id,target,fold
0,8bd42737-b140-4db8-af4c-59839922129a,0,8
1,7d2e034a-3270-4429-aeac-418c46bf0d7d,1,0
2,c1b97bb6-03d2-4027-8d54-0dd64a927a6c,0,4
3,883528e9-0599-4d30-9f4c-1daf9cf74082,1,3
4,7fd1210c-b1c0-4bdf-b190-7bfa10fe51d2,0,1
...,...,...,...
22019,759b16f2-f527-4f04-85ba-682fd1af8a89,0,4
22020,2e8f5550-d794-411c-b36e-d277e88f5d30,1,5
22021,3ddc9eae-ba7b-4dab-8e43-5c188daf9692,0,0
22022,b7132b2c-5ca4-44ac-b4e7-07e43d4b7547,1,4


Chile coordinates: fold 1/10, accuracy=0.80163, seconds=24.7


Chile coordinates: fold 2/10, accuracy=0.81434, seconds=25.2


Chile coordinates: fold 3/10, accuracy=0.79936, seconds=25.9


Chile coordinates: fold 4/10, accuracy=0.80708, seconds=25.7


...

,fold,n_train,n_test,accuracy,recall_left,recall_right,auc,fit_seconds
0,0,445,50,0.860000,0.921053,0.666667,0.912281,1.377983
1,1,445,50,0.880000,0.921053,0.750000,0.911184,1.377110
2,2,445,50,0.840000,0.947368,0.500000,0.955044,1.381929
3,3,445,50,0.860000,0.918919,0.692308,0.956341,1.396351
4,4,445,50,0.820000,0.891892,0.615385,0.914761,1.413326
5,5,446,49,0.857143,0.891892,0.750000,0.951577,1.379276
6,6,446,49,0.877551,0.864865,0.916667,0.897523,1.383263
7,7,446,49,0.816327,0.864865,0.666667,0.911036,1.390964
8,8,446,49,0.755102,0.810811,0.583333,0.811937,1.411909
9,9,446,49,0.857143,0.918919,0.666667,0.925676,1.376712


,fold,PC1
0,0,1.0
1,1,1.0
2,2,1.0
3,3,1.0
4,4,1.0
5,5,1.0
6,6,1.0
7,7,1.0
8,8,1.0
9,9,1.0


,FPR,TPR,threshold
0,0.000000,0.000000,inf
1,0.008043,0.221311,1.0000
2,0.008043,0.245902,0.9995
3,0.010724,0.278689,0.9990
4,0.013405,0.295082,0.9985
...,...,...,...
88,0.276139,0.942623,0.0030
89,0.278820,0.950820,0.0025
90,0.281501,0.950820,0.0015
91,0.292225,0.983607,0.0005


{'coordinates': {'features': ['z1', 'z2'],
  'fold_accuracy_min': 0.8163265306122449,
  'fold_accuracy_max': 0.9387755102040817,
  'fold_accuracy_mean': 0.8725714285714286,
  'oof_accuracy': 0.8727272727272727,
  'oof_recall_left': 0.9222520107238605,
  'oof_recall_right': 0.7213114754098361,
  'oof_auc': 0.9317672394849033,
  'confusion_counts_left_right': [[344, 29], [34, 88]],
  'confusion_row_normalized_left_right': [[0.9222520107238605,
    0.0777479892761394],
   [0.2786885245901639, 0.7213114754098361]],
  'mean_feature_importance': {'z1': 0.7639777377918814,
   'z2': 0.23602226220811867}},
 'demographics': {'features': ['age', 'sex', 'region'],
  'fold_accuracy_min': 0.64,
  'fold_accuracy_max': 0.8,
  'fold_accuracy_mean': 0.737469387755102,
  'oof_accuracy': 0.7373737373737373,
  'oof_recall_left': 0.9517426273458445,
  'oof_recall_right': 0.08196721311475409,
  'oof_auc': 0.5283039599173736,
  'confusion_counts_left_right': [[355, 18], [112, 10]],
  'confusion_row_normalized

,country,model,n,fold_accuracy_min,fold_accuracy_max,fold_accuracy_mean,oof_accuracy,oof_recall_left,oof_recall_right,oof_auc
0,Chile,coordinates,22024,0.790191,0.815622,0.803305,0.803305,0.837586,0.758628,0.867072
1,Chile,demographics,22024,0.545166,0.579210,0.561160,0.561161,0.715615,0.359862,0.567266
2,Chile,combined,22024,0.801090,0.831971,0.815519,0.815519,0.849703,0.770968,0.879207
3,Chile,pc1,22024,0.725828,0.755677,0.743871,0.743870,0.771225,0.708220,0.837743
4,France,coordinates,788,0.860759,0.949367,0.912496,0.912437,0.959627,0.701389,0.941808
5,France,demographics,788,0.810127,0.822785,0.817267,0.817259,1.000000,0.000000,0.542912
6,France,combined,788,0.860759,0.949367,0.911246,0.911168,0.961180,0.687500,0.945798
7,France,pc1,788,0.797468,0.949367,0.879552,0.879442,0.930124,0.652778,0.901953
8,Brazil,coordinates,495,0.816327,0.938776,0.872571,0.872727,0.922252,0.721311,0.931767
9,Brazil,demographics,495,0.640000,0.800000,0.737469,0.737374,0.951743,0.081967,0.528304


## Describe validation at different observed participation levels

Merge the combined model's live out-of-fold predictions with each person's total retained comparison count. `qcut` forms country-specific groups by quantiles of that count. Ties stay together, so duplicate quantile edges can yield fewer than the requested ten groups. The reported minimum and maximum counts show what each group contains.

Every eligible participant must appear in exactly one group, and group error counts must sum to the overall error count. These are descriptive comparisons of existing histories. They do not truncate choice histories or estimate the causal effect of asking more questions.


In [4]:
activity_frames = []
for country in COUNTRIES:
    destination = OUT / country
    count_path = ROOT / COUNTS[country]
    counts = pd.read_csv(count_path, dtype={"uuid": str, "user_id": str})
    counts = counts.rename(
        columns={
            "uuid": "participant_id",
            "user_id": "participant_id",
            "total_choices": "observed_count",
            "n_choices": "observed_count",
        }
    )
    assert counts.participant_id.is_unique
    assert (counts.observed_count > 0).all()
    assert (counts.observed_count == counts.observed_count.astype(int)).all()
    prediction = combined_predictions[country].copy()
    prediction = prediction.merge(
        counts[["participant_id", "observed_count"]],
        on="participant_id",
        how="left",
        validate="one_to_one",
    )
    assert prediction.observed_count.notna().all()
    prediction["group"] = pd.qcut(
        prediction.observed_count, ACTIVITY_GROUPS, labels=False, duplicates="drop"
    )
    assert prediction.group.notna().all()
    prediction["group"] = prediction.group.astype(int) + 1
    rows = []
    for group, frame in prediction.groupby("group"):
        rows.append(
            {
                "country": country,
                "group": int(group),
                "n": len(frame),
                "n_left": int(frame.target.eq(0).sum()),
                "n_right": int(frame.target.eq(1).sum()),
                "count_min": int(frame.observed_count.min()),
                "count_median": float(frame.observed_count.median()),
                "count_max": int(frame.observed_count.max()),
                "accuracy": float(accuracy_score(frame.target, frame.prediction)),
                "recall_left": float(
                    recall_score(frame.target, frame.prediction, pos_label=0, zero_division=0)
                ),
                "recall_right": float(
                    recall_score(frame.target, frame.prediction, pos_label=1, zero_division=0)
                ),
                "error_count": int(frame.target.ne(frame.prediction).sum()),
            }
        )
    activity = pd.DataFrame(rows)
    assert activity.n.sum() == len(prediction)
    assert activity.error_count.sum() == prediction.target.ne(prediction.prediction).sum()
    assert np.allclose(activity.accuracy, 1 - activity.error_count / activity.n)
    display(prediction)
    prediction.to_csv(destination / "combined_oof_with_activity.csv", index=False)
    display(activity)
    activity.to_csv(destination / "activity_groups.csv", index=False)
    activity.to_latex(destination / "activity_groups.tex", index=False, float_format="%.6f")
    activity_frames.append(activity)

all_activity = pd.concat(activity_frames, ignore_index=True)
display(all_activity)
all_activity.to_csv(OUT / "activity_groups.csv", index=False)
all_activity.to_latex(OUT / "activity_groups.tex", index=False, float_format="%.6f")


,participant_id,target,fold,prediction,probability_right,observed_count,group
0,8bd42737-b140-4db8-af4c-59839922129a,0,8,1,0.8710,63,6
1,7d2e034a-3270-4429-aeac-418c46bf0d7d,1,0,1,0.8315,21,2
2,c1b97bb6-03d2-4027-8d54-0dd64a927a6c,0,4,1,0.6230,21,2
3,883528e9-0599-4d30-9f4c-1daf9cf74082,1,3,0,0.0280,91,8
4,7fd1210c-b1c0-4bdf-b190-7bfa10fe51d2,0,1,0,0.0850,60,6
...,...,...,...,...,...,...,...
22019,759b16f2-f527-4f04-85ba-682fd1af8a89,0,4,0,0.2845,91,8
22020,2e8f5550-d794-411c-b36e-d277e88f5d30,1,5,1,0.9930,114,9
22021,3ddc9eae-ba7b-4dab-8e43-5c188daf9692,0,0,1,0.5705,228,10
22022,b7132b2c-5ca4-44ac-b4e7-07e43d4b7547,1,4,1,0.7510,29,4


,country,group,n,n_left,n_right,count_min,count_median,count_max,accuracy,recall_left,recall_right,error_count
0,Chile,1,2640,1258,1382,2,19.0,20,0.774242,0.795707,0.754703,596
1,Chile,2,3497,2161,1336,21,21.0,21,0.770947,0.826932,0.680389,801
2,Chile,3,647,385,262,22,22.0,23,0.788253,0.844156,0.706107,137
3,Chile,4,2076,1243,833,24,28.0,33,0.793353,0.839903,0.723890,429
4,Chile,5,2237,1378,859,34,40.0,47,0.820742,0.859942,0.757858,401
5,Chile,6,2145,1216,929,48,56.0,66,0.823776,0.851151,0.787944,378
6,Chile,7,2211,1242,969,67,79.0,85,0.838987,0.855072,0.818369,356
7,Chile,8,2178,1188,990,86,95.0,102,0.841598,0.881313,0.793939,345
8,Chile,9,2190,1220,970,103,118.0,145,0.851598,0.879508,0.816495,325
9,Chile,10,2203,1171,1032,146,200.0,764,0.866092,0.879590,0.850775,295


,participant_id,target,fold,prediction,probability_right,observed_count,group
0,e9001e65-7aa1-468d-a7c6-dca4ad023e59,0,9,0,0.0120,41,1
1,41e3bf3a-e792-4521-8d3a-8f974f329345,0,9,0,0.0000,3781,10
2,a664ff56-e303-4ab5-8841-bc950222e75f,0,9,0,0.0000,2911,9
3,7490cb32-18ba-4572-9310-c0c3801438d0,1,9,1,0.9810,2776,9
4,fecf3440-84f3-4bcb-b2c4-f6c939d80ec7,0,4,0,0.0050,2457,8
...,...,...,...,...,...,...,...
783,9026410e-9aa7-4bf8-aafa-8dbd26c94c63,1,1,1,0.6330,220,2
784,67e9a740-4137-4834-9331-a54929c5e6a9,0,1,0,0.0000,1720,5
785,48243d8e-ab20-4379-b9ba-ba76c4cde2f3,0,4,0,0.0635,1279,4
786,a7adb7e7-e526-49b9-93d5-563f75aa7714,0,6,0,0.0000,2101,6


,country,group,n,n_left,n_right,count_min,count_median,count_max,accuracy,recall_left,recall_right,error_count
0,France,1,79,61,18,14,41.0,81,0.835443,0.868852,0.722222,13
1,France,2,79,62,17,82,130.0,220,0.911392,0.983871,0.647059,7
2,France,3,79,62,17,223,369.0,577,0.886076,0.951613,0.647059,9
3,France,4,78,67,11,588,1055.5,1428,0.948718,1.000000,0.636364,4
4,France,5,79,70,9,1433,1645.0,1856,0.911392,0.971429,0.444444,7
5,France,6,80,69,11,1863,2038.0,2164,0.900000,0.942029,0.636364,8
6,France,7,77,62,15,2168,2332.0,2454,0.987013,1.000000,0.933333,1
7,France,8,79,66,13,2455,2574.0,2698,0.898734,0.969697,0.538462,8
8,France,9,79,60,19,2700,2808.0,3007,0.936709,0.966667,0.842105,5
9,France,10,79,65,14,3008,3263.0,3809,0.898734,0.953846,0.642857,8


,participant_id,target,fold,prediction,probability_right,observed_count,group
0,e3428c81-c1a3-4139-9deb-ab2b9aae481d,0,8,0,0.0080,20,2
1,248860f8-a008-4207-84fd-a88cdc02e3f1,0,0,0,0.3175,114,6
2,f3d2061a-9ffa-4fc2-bf46-f9990b6055ec,0,0,0,0.0150,25,3
3,457cc0fc-957a-4984-a685-587b0686d18d,0,8,0,0.0670,40,4
4,fc8481a9-7975-43f3-8a76-c054d51e73ea,1,5,1,0.7850,772,10
...,...,...,...,...,...,...,...
490,33c490b5-f5e4-4db9-8d77-4f76fd0d363f,0,2,0,0.1765,52,5
491,5d5f537f-a895-49ed-966d-a5b31539155d,1,5,1,0.8365,475,8
492,035fef84-952f-40cf-b6bc-59419ebc4234,1,1,1,0.8140,470,8
493,730666d4-bf22-4c25-9df0-cd447cd72100,1,3,1,0.8250,25,3


,country,group,n,n_left,n_right,count_min,count_median,count_max,accuracy,recall_left,recall_right,error_count
0,Brazil,1,53,35,18,1,11.0,14,0.830189,0.885714,0.722222,9
1,Brazil,2,47,34,13,15,18.0,21,0.808511,0.823529,0.769231,9
2,Brazil,3,49,39,10,22,26.0,33,0.897959,0.974359,0.600000,5
3,Brazil,4,51,42,9,34,44.0,51,0.862745,0.904762,0.666667,7
4,Brazil,5,49,33,16,52,63.0,72,0.857143,0.878788,0.812500,7
5,Brazil,6,48,33,15,73,110.0,190,0.854167,0.939394,0.666667,7
6,Brazil,7,49,42,7,194,299.0,357,0.918367,0.928571,0.857143,4
7,Brazil,8,50,41,9,363,428.5,495,0.960000,0.951220,1.000000,2
8,Brazil,9,49,42,7,496,537.0,684,0.918367,0.928571,0.857143,4
9,Brazil,10,50,32,18,686,800.0,1192,0.980000,1.000000,0.944444,1


,country,group,n,n_left,n_right,count_min,count_median,count_max,accuracy,recall_left,recall_right,error_count
0,Chile,1,2640,1258,1382,2,19.0,20,0.774242,0.795707,0.754703,596
1,Chile,2,3497,2161,1336,21,21.0,21,0.770947,0.826932,0.680389,801
2,Chile,3,647,385,262,22,22.0,23,0.788253,0.844156,0.706107,137
3,Chile,4,2076,1243,833,24,28.0,33,0.793353,0.839903,0.723890,429
4,Chile,5,2237,1378,859,34,40.0,47,0.820742,0.859942,0.757858,401
5,Chile,6,2145,1216,929,48,56.0,66,0.823776,0.851151,0.787944,378
6,Chile,7,2211,1242,969,67,79.0,85,0.838987,0.855072,0.818369,356
7,Chile,8,2178,1188,990,86,95.0,102,0.841598,0.881313,0.793939,345
8,Chile,9,2190,1220,970,103,118.0,145,0.851598,0.879508,0.816495,325
9,Chile,10,2203,1171,1032,146,200.0,764,0.866092,0.879590,0.850775,295
